# Flash attention
## 搭建环境
在ubuntu虚拟机上搭建了环境，可以正确编译以及cuda程序，实现cuda程序的自动跳转，方便编写cuda代码
因此编写代码就放在虚拟机上

因为flash attention2 3 需要利用特殊的硬件特性，因此需要在租相应的平台，在平台上搭建环境很麻烦

先看看autodl，能不能采用docker的方式运行
## 实现flash attention
### 实现层次
需要先看看怎么实现，应该是直接实现flash attention 算子，包括对应的forward、backward

上层的transfomer 模块相应的调用算子（看一下以前的transformer实现）

最后确认了实现的层次：Ops -> CUDA Kernel -> Pybind -> Op Wrapper -> Module Call
### 实现方法
应该用什么实现呢？cuda、triton、cuTile

使用cuda编程的话，应该借用cutblass模版库进行编程

如果使用triton实现的话，绕过了pybind，但是将cuda后端管理的ptr给triton模块，然后再在triton层面进行编程

使用cuTile编程的话，依旧是python编程，和triton类似

### 测试程序
需要先编写测试程序，首先是flash attention 算子的测试程序，该测试程序需要包括一下几个方面

1.直接调用cuda程序中的实现进行测试（确定在ops层面进行测试）

2.需要验证算子的正确性，那和什么进行比对呢（或者在python层面进行比对正确性，参考一下以前的比较）

3.每个测试应该进行多次迭代，从而可以测试时间，包括不同seq长度

（可以参考官方的flash attention实现，看看它是怎么进行测试的）





接下来要做的事：

在autodl平台上跑一次

了解cuTile，决定是用cutlass还是用cuTile

看看别人的实现以及怎么测试、怎么benchmark

backward 不应该都是tensor操作吗，直接调用Narray api不就构建不了计算图了吗？（需要构建一个ops用来计算backward）

测试的时候是调用ops测试，还是module呢，以及怎么获得时间


In [ ]:
# !pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git

# Download the PTB dataset

import urllib.request
import os

!mkdir -p './data/ptb'
# Download Penn Treebank dataset
ptb_data = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/ptb."
for f in ['train.txt', 'test.txt', 'valid.txt']:
    if not os.path.exists(os.path.join('./data/ptb', f)):
        urllib.request.urlretrieve(ptb_data + f, os.path.join('./data/ptb', f))

In [ ]:
!make clean
!make -j1

In [ ]:
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd

In [ ]:
# import sys
# sys.path.append('./python')

## 测试
### 正确性验证
如果在NDArray层面去测试的话，需要借用numpy来进行比较，而numpy没有现有的api计算flash attentionn，需要根据已知的qkv计算self attention，因此需要创建numpy的qkv，然后根据self attention的定义去计算，再与cuda 的NDArray计算结果进行比较，比较麻烦。

因此本测试在flash attention module层面进行正确性验证，仿照已有测试中的attention_activation 测试编写，将flash attention的结果与已知的label进行对比，同时编写了新的测试将结果和torch的flash attention 计算结果对比，在与torch对比的测试用例 sequence length比较长，符合实际。
### benchmark
对于benchmark说，需要计算TFLOPS，为了最贴近计算，采用算子层面进行benchmark，先对GPU进行预热，flashattention进行计算，迭代50次，计算时间以及总的操作数，从而计算TFOPS。当前的计算基于causal = false，dropout = 0

In [ ]:
# !python3 -m pytest tests/hw4/test_transformer.py -l -v -k "attention_activation_vs_torch"

In [ ]:
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and 64 and False and 0.0 and cuda"

In [ ]:
!python3 -m pytest tests/project/test_flashattention.py  -s -l -v -k "test_attention_activation_vs_torch and 5 and 1024 and 64 and False and 0.0 and cuda"

In [ ]:
# # stub hook 验证
# import sys
# !{sys.executable} -m pytest tests/project/test_flashattention_stub.py -l -v

In [ ]:
# baseline
!python3 tests/project/benchmark_torch.py \
  --batch_size 8 \
  --num_heads 12 \
  --seq_len 1024 \
  --head_dim 64 \
  --dropout 0.0 \
  --backend flash \
  --profile

##  --backend cudnn
#   --backend mem_efficient
#   --backend math

In [ ]:
!python3 tests/project/benchmark.py \
  --batch_size 8 \
  --num_heads 12 \
  --seq_len 1024 \
  --head_dim 64 \
  --dropout 0.0 \
  --warmup 10 \
  --repeats 100

In [ ]:
# ncu profile
!ncu --launch-skip 10 --launch-count 1 --set full \
  --kernel-name regex:flash_attention_kernel --kernel-name-base demangled \
  --import-source yes --source-folders /home/xyx/needle/src,/home/xyx/needle/python \
  -o /home/xyx/needle/flash_attention_profile -f \
  python3 /home/xyx/needle/tests/project/benchmark.py \
  --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0 >> profile.txt

## stub
对于ops层面的stub来说，是返回tensor tuple还是tensor呢？
* 需要查看probs对于后续有没有什么作用，如果没有作用可以直接返回result tensor
* 如果返回tensor tuple，是backend计算直接返回tensor tuple还是在ops层进行组装后返回，需要参考stack的实现，返回tensor tuple后进行第i个结果的取用会不会产生额外的开销

最后决定只是返回result，因此定义为tensorops


## 调用层次关系
module调用ops进行计算，ops的forward计算的是NDArray，使用NDArray定义的array api进行计算，NDArray的backend device不同，因此调用进不同的device function。

## 精度
在pytorch中的attention的实现中，因为A100的tensor core只接受输入的形式为 fp16/bf16/tf32，不支持fp32的输入，因此没有开启混合精度的情况下，是没有使用flash attention的实现的，而使用的是原生的的其他优化过的kernel

如果开启混合精度模式，计算的流程如下：QKV（fp32）加载到sram中cast 为（fp16），使用tensor做矩阵乘法时，结果accumulator保存为fp32。进行softmax的时候，为了防止溢出保持fp32，再cast为fp16与 V（fp16）相乘，
最后的结果O为fp16（为了节省sram 写到 GMEM中的带宽）。

现在我自己的实现最好确定输入的类型时fp16，那这样其他实现的cuda计算就用不了了，一个weired解决办法是输入继续保持为fp32但是计算的结果o为fp32再重新cast为fp16。

* 看看别人的flash attention怎么实现的
    * 别人的flash attention中没有使用cute模版编程，只是借用了cutlass的gemm实现来优化自己的设计。

* 如果在我的实现中使用cutlass，应该怎么使用，和cute有区别吗？
    * CuTe 强大的 Layout Algebra (布局代数) 能够让你在不陷入指针算术泥潭的情况下，优雅地处理复杂的 Tensor Core 数据映射、Shared Memory Swizzle 和 Bank Conflict。


1. 现在需要研究cute 的核心，使用cute 模版编程来实现自己的flash attention。




 ## flash attention with mma 
在navie 实现上做了这些优化：
1. 传输优化
* global -> share memory的传输使用了指令 **cp.async.ca.shared.global.L2::128B** ,该指令每个线程每次传输128bit的数据，因此传输的数据也需要满足相应的要求，即如果传输的数据是fp16(m, n)，在n维度上对应的数据物理地址要求每8个元素一组连续，8个元素与另一组则没有要求，且他是异步传输指令。
* 同时对于线程的排布采用了**Layout<Shape<_16, _8>, Stride<_8, _1>>** row major的方法，因为global memory中源数据是row major储存的，这样处理的话，多个线程访问global memory的transaction会合并为一个transaction。
2. 矩阵计算优化
* matrix计算从最原始的使用FMA改为了使用**mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32**，与FMA不同的是，这里的矩阵乘法是以warp为单位进行计算的，每个thread register内需要包含A B C D相应的fragment。在make tiled mma的时候，第二个参数thr_layout，如果在K维度上有排列，举出这样一个例子便于理解：（Layout<Shape<_2, _1, _2>>{}）调用gemm后，那么对于(m, n)的结果矩阵C，前1/2 K维度的结果保留在warp 0、1，后1/2 K维度的结果保留在warp 2、3中，因此需要再进行一次规约才能得到最后的结果矩阵。
* gemm api调用时，需要满足这样的假设：A、B的逻辑形状分别是(m, k) (n, k)，这样gemm才知道分得清维度m、n、k。因此在计算mma2时，对sV进行了转置再传给gemm。至于对于采用的atom **SM80_16x8x8_F32F16F16F32_TN** 最后面是T还是N，其实和逻辑矩阵的形状没有关系，这里的TN影响的是最后thread中fragment到底是哪一部分，cute TV value为我们处理了。
3. atom -> tiled -> all（以mma举例解释，copy指令差不多）
* atom 即一个warp指令，通过make tiled，使用多个warp计算或者一个warp计算更多的值形成tiled mma。
* 一个tiled mma对A、B、C进行partition，对于结果矩阵来说使用tiled mma的逻辑矩阵（m，n）平铺整个大的（M，N），即在row、column方向上分别进行除法，产生了新的维度。这里的底层逻辑还是通过一个warp计算更多的值实现的。
4. 如何实现不需要share memory的过渡，mma1储存在register中的accumulator C直接用于mma2的operator A？
* 每个register是每个thread私有的，因此这就要求形成tiled mma最后通过partition形成的fragment，对于每个thread来说拥有的mma2 accumulator C的值刚好可以用于mma1 operator A。
* 第一层即选用atom，选用的atom满足**ALayout = CLayout**，这样在一个atom内部，mma1的C fragment即mma2的A fragment
* 第二层即形成tiled mma，mma1 C和mma2 A共有的维度是m，线程的排布必须都集中在该维度上，否则就会出现fragment大小不一致的情况
* 第三层即partition，对于partition的切分来说，满足上面的两个条件后，切分后的fragment相同是自然的，因为切分即一个warp计算更多的值实现，而且切分的逻辑都是对逻辑矩阵中的row column进行切分
5. 对于不采用share memory过渡P的方法，如何计算row max以及row sum呢？
* fragment是thread私有的，以及一个thread的数据跨越不同的row，以及一个row的数据包含在多个thread中，这就要求我们清晰的明白TV layout，通过print_latex可以得到tiled_mma的(m, n) -> (T, V)，从而推断出thread的fragment
* 因为一个row的数据储存在相同warp下的多个thread中，因此采用指令__shfl_xor_sync进行规约，最后的结果是每个thread都有他的那部分row的row max和row sum，因为mma1 fragment C = mma2 fragment A = mma2 fragment C的递进关系，这里的row max又可以用来rescale output O


还可以做的优化：
1. share memory -> register 采用ldmatrix
2. share memory swizzled
3. 最后在register中的output直接copy到global中，或者lshare memory再copy到global中怎么优化
4. software pipeline，怎么进行copy和compute overlap


## ldmatrix optimization
对于QKV 从share memory到 register的拷贝使用了ldmatrix命令，使用时有一下几点需要注意：

```ptx
ldmatrix.sync.aligned.m8n8.num{.trans}{.ss}.type r, [p];
ldmatrix.sync.aligned.m8n16.num{.ss}.dst_fmt.src_fmt r, [p];

.num = {.x1, .x2, .x4};
.ss = {.shared{::cta}};
.type = {.b16, .b8};
.dst_fmt = { .b8x16 };
.src_fmt = { .b6x16_p32, .b4x16_p64 };
```
1. 该指令也是warp level的指令，配合mma一起使用
2. 如何挑选num字段呢？
    * 需要结合形成的tiled mma来指定num字段。比如在代码中对于矩阵Q来说，形成的tiled mma每一个warp的矩阵大小是16x8，因此这里选用的num为x2（当然你也可以选择更小的atom，那么在形成tiled copy的时候对应的就是发起更多次的操作）
    * 相应的x1为8x8矩阵，x2为2个8x8矩阵，x4为4个8x8矩阵
3. 如何决定是否transpose？
    * 首先需要明白的一点是，是否transpose与每个register中包含逻辑矩阵的哪些坐标是不相关的，后者只和选用的mma atom相关，，而copy可以从mma atom中得到信息，来知道怎么样进行copy。无论是transpose还是not transpose，最后对于逻辑矩阵的TV layout都是不会改变的
    * 是否需要transpose，参考的是逻辑坐标与物理坐标之间的映射关系。以sQ进行举例，sQ为（m，k）：（k，1），他是row major的，因此需要使用not transpose的ldmatrix进行copy
4. 如何正确的进行ldmatrix分为一下几个步骤：
    * 根据share memory layout逻辑坐标与物理坐标的关系选用atom是否transpose
    * 分析构造的tiled mma中每个warp需要多大的matrix来确定aomt size大小
    * 使用tiled mma以及copy atom来构造tiled copy，此时cute会帮我们自动分析出每个thread share memory的pointer

形成的tiled copy TV layout在ldmatrix.pdf中，可以看到最后destination的TV layout和各自tiled mma所要求的TV layout完全一样（需要注意的是mma的operator B打印的方式是(k, n),而copy 打印的方式都是(n, k)因此看operator B的时候需要转置一下）

## output O share memory to global optimization
对于最后的结果矩阵O，优化前，采用每个thread每次搬运一个sO到gO中，由于L1到L2的传输是按照sector，因此每个thread每次传输2B的sO到gO中，事实上会造成32B sector的传输。也就是说每次传输32B的数据只有2B的数据真正被用上。因此采用了universal copy<uint128> 指令进行数据传输，优化有两方面：

1. 每个thread每个指令运输128b的数据，即8个half
2. 利用thread的排布，形成coalesce，对于global memory的store，多个transaction会合并为一个transaction


## share memory swizzle
对于global -> share memory路径来说，没有swizzle的版本并没有发生bank conflict，需要对我以前对bank conflict的理解做以下纠正：

1. 一个warp内的所有active lane执行某一load 或者store指令产生大于两个wavefront，不一定是因为bank conflict，有可能是因为本来就需要两个wavefront，举出这样的例子进行解释：
    * 对于global -> share memory路径来说，没有swizzle的版本并没有发生bank conflict，但是产生了4个wavefront。以128bit的形式读写共享内存，此时线程需要访问的单位数据量为16byte，32个线程需要访问的数据量为16byte x 32 = 512byte。完整的512byte需要4个phase才能完成访问，第一phase，T0-T7无bank conflict的访问所有bank，第二phase，T8-T15无bank conflict的访问所有bank，第三phase，T16-T23无bank conflict的访问所有bank，第四phase，T24-T31无bank conflict的访问所有的bank。这四个phase就是四个wavefront
2. 因此需要优化的多余的wavefront是bank conflict造成的，对于软件来说是发起一个request，硬件会产生一个work package，硬件会知道需要访问哪些地址，在尽可能在一个wavefront中访问多的bank free的数据，无法合并的且没有包含全部32bank的wavefront即bank conflict产生的

真正需要优化的share memory load是ldmatrix：

1. 在没有swizzle的情况下，ldmatrix使用的是T0-8进行load（所有threads参与的是到fragment register的过程），8个thread访问8 row的数据（都属于bank0 - 4），因此产生的 8 way bank conflict
2. 使用原始的layout和swizzle<B, M, S>。Swizzle定义了三个参数: B、M、S。它们共同表达描述一维坐标向二维空间映射的三个层次。当我们把一个一维度坐标转换成二维坐标时，我们首先将一维中连续的几个元素作为新空间中的基础元素，然后描述该二维空间有多少行和列。其中一维坐标中连续的 $2^M$ 个元素构成二维空间中最基本的元素， $2^S$ 表示新的二维空间中有多少列， $2^B$ 表示新的二维空间中有多少行。在我的实现中B M S都取3，即8个数据为一个element，分为8个column进行swizzle，每8个row进行一次循环
3. sV_trans layout比较有意思，对sV进行了swizzle后，无法简单的交换两个mode的shape以及stride得到sV_trans。现在我需要做的是得到新的view 即sV_trans满足下面的要$sV\_trans(d,k) == sV(k,d)$ ，结合composition的定义，即对原来的sV_tans进行相同swizzle的composition即可。



## software pipeline
因为global memory -> share memory的过程是async的，因此将该数据传输阶段与其他阶段进行pipeline，但是gemm、online softmax、share memory -> register的过程是阻塞的，因此这些阶段都是顺序进行的。使用software pipeline，进行了以下overlap

1. 在同一次迭代内，gemm0 与 copy V进行overlapped，先发起V global memory -> share memory的cp.async，再进行gemm0
2. 两次迭代间，迭代i gemm1 与迭代i+1需要的copy K进行overlapped，即在迭代0 gemm1开始前，发起下一次迭代需要的copy $K_{i+1}$ global memory -> share memory

## performace
configuration：64 head dimension，1024 sequence length
### speed of light
* A100 312 TFLOPS
* H100 SXM 989.5 TFLOPS
### A100系列
* flash attention 2 paper：153(TFLOPs/s) A100 80GB SXM4
* 实测pytorch中的_scaled_dot_product_flash_attention: 150.058(TFLOPs/s) A800 80GB PCIE
* 实测pytorch中的_scaled_dot_product_cudnn_attention: 158.142(TFLOPs/s) A800 80GB PCIE
* 自己实现的flash attention：97.218(TFLOPs/s) A800 80GB PCIE
### H100系列
* flash attention 3 paper：flash attention 2 306(TFLOPs/s) H100 80GB SXM5
* flash attention 3 paper：cudnn attention 2 373(TFLOPs/s) H100 80GB SXM5
* 实测pytorch中的_scaled_dot_product_flash_attention: 268.956(TFLOPs/s) H100 80GB SXM5
* 实测pytorch中的_scaled_dot_product_cudnn_attention: 426.311(TFLOPs/s) H100 80GB SXM5
* 自己实现的flash attention：164.515(TFLOPs/s) H100 80GB SXM5


## compile optimization
编译时通过加入相应的flag，提示nvcc进行更为激进的优化，增加了以下flag：
* -O3 
* --use_fast_math 使用更快但精度较低的数学函数，并启用近似浮点行为。它大致包含一些 fast math 设置，比如快速除法、快速 sqrt、低精度 transcendental functions。
* -Xptxas -O3  把 -O3 传给 PTX assembler，让底层汇编优化更积极。

performance 提升：
* A800: 97.218 -> 104.094
* H100: 164.515 -> 173.837

## else optimization
1. 删除了多余的__syncthreads(); 对于sK -> rK后是不需要整个block尽心sync的，因为register是每个thread私有的，而且无论是ldmatrix还是mma都是warp级别的操作
ldmatrix加载的数据即mma所需要的，因此对于一个warp来说只有ldmatrix完成后mma才会继续下面的计算 (不成立，该__syncthreads()，放置sK_{i}被sK_{i + 1}覆盖)
* A800 104.094 -> 106.798（x）
2. 构造tiled_mma时在n维度上从_8扩展到_64。猜想gemm调用tiled_mma时，如果是n = _8，他会在k维度上先进行mma atom的调用，但是在k维度上的规约会有依赖关系；如果tiled_mma的n = _64，gemm先计算的是tiled_mma，即在n维度上进行mma_atom的调用，数据前后没有依赖关系，所以performance提升 (最后profile的结果显示是因为register使用减少了，theoretical occupancy 增大，从每个sm 2 block变成了 每个sm 3 block)
* A800 104.094 -> 118.461
* H100 173.837 -> 190.656
3. 因为rO -> sO 会造成bank conflict，因此直接rO -> gO，但是performance降低。因为rO -> gO无法使用大字节的数据传输（即一个thread copy 128bit）
* A800 118.461 -> 111.314
* H100 190.656 -> 179.463
4. 在tiled mma中无法使用permutation将rO对应为连续的sO value的原因
* permutation是对sV_trans进行permutation，但是sV_trans是column major，对应的ldmatrix指令是thread 0 - 4填入每一个column的地址，然后在加载该column连续的8个数据到相应的线程中；同时permutation也是在n维度上进行重排，重排后，一次mma所需要的matrix在n维度上就不连续了，因此与ldmatrix的layout不符。
5. 使用了更大的mma atom，将两个mma atom从SM80_16x8x8_F32F16F16F32_TN 改成了 SM80_16x8x16_F32F16F16F32_TN，因为每次调用tensor计算了更大的matrix，减少了开销，因此获得了矩阵计算的更大吞吐，performance得到了提高。同时因为形状的改变做了以下修改：
* 需要确认对于逻辑矩阵P来收tiled mma1得到的accumulator以及tiled mma2需要的operator A，两者每个线程元素相同
* 对于register来说，mma1的accumulator mma1rP为ptr[32b](0x7fc06bfffc60) o ((_2,_2),_1,_8):((_1,_2),_0,_4)，对应为（mma1， m， n）；mma2的operator A mma2rP为 ptr[16b](0x7fc06bfffc20) o ((_2,_2,_2),_1,(_2,_2)):((_1,_2,_4),_0,(_8,_16))，对应为（mma2， m， n），可以看到layout不同，但是拥有的元素个数相同，且通过上一条的验证拥有的元素元素相同，且物理排序都相同，因此只需要将mma1rP.data + mma2rP.layout，组成mma2所需要的tensor
* A800 118.461 -> 128.358
* H100 190.656 -> 207.152
6. 对于register O -> share O 来说，发生了8 way bank conflict，因此需要对share memory进行swizzle
* 对于share memory来说，发生bank conflict是在warp范围内，每次warp执行register O -> share O 操作的数据大小是8 * 8 的矩阵，因为8 row在同一个column，即都在bank 0 - 4，会发生8 way bank conflict。因此在选取swizzle的参数时，基本的元素大小为8，进行8个column的swizzle
* A800 128.358 -> 131.966
* H100 207.152 -> 211.834
7. 使用了更大的ldmatrix，即SM75_U32x4_LDSM_N 调用一次指令可以加载4个8x8的矩阵
* 此时的tiled mma1 和tiled mma2都是64 x 64 x 16，每个warp加载的矩阵为 8*2 个 8 x 8 大小的矩阵，因此通过使用copy atom以及 tiled mma形成 tiled copy后通过get_slice，每个线程获得自己的share memory坐标
* H100 211.834 -> 231.179
* A800 131.966 -> 133.322
8. double buffering
* 使用double buffering实现了更好的compute、data transfer overlap，之前的software pipeline方法可以看到在__syncthreads()仍然有很多stall 在barrier，即data transfer没有完成；现在采取的策略是，在迭代一开始就发起下一次迭代所需要的sK sV，因此所需要的share memory分为2个stage，对这两个share memory的read write交替进行，故为double buffering
* A800 133.322 -> 143.862
* H100 231.179 -> 255.154

最后在A800上达到了官方实现的95.87%

## H100 optimization
### wgmma
wgmma 是 hopper架构的推出的异步mma指令，与A100架构的mma有这些不同：
1. 执行的范围是warp group（4 warp， 128threads），更多的warp参与执行该wgmma，在执行该指令时会同时占用sm中的4个tensor core，因此每次执行可以支持更大的矩阵打下（64x64x16 vs 16x16x8），提供了更大的矩阵计算吞吐
2. 对于operand A来说，需要在share memory中，operand B可以选择在share memory或者register中，accumulator C在register中，对于operand 在share memory中可以减少register的使用。如果在share memory中则需要符合wgmma特定的layout，cute中提供了特定的layout atom来构造share memory的layout。对于wgmma指令来说 operand在share memory的情况，partition_and_fragment构造的是descriptor传给wgmma指令，每个thread所构造的descriptor是相同的（这和mma 每个thread含有不同的fragment不同），同时wgmma理解descriptor是按照core matrix理解的，因此share memory的layout的要求则需要符合core matrix，不同的swizzle方法，core matrix交错方式不同
    > 1. wgmma atom可以分成多个core matrix，core matrix大小为1x8（这里的1代表128bits），逻辑上代表128bits/sizeof(T) x 8的小矩阵
    > 2. （讨论k major情况）在no swizzle的情况下，每个core matrix在share memory中连续，多个core matrix tile_to_shape组成了wgmma atom；对于32B swizzle的情况，则一个swizzle atom大小为2x8（2个core matrix），并且在share memory中按照row交错排列，再在此基础上进行swizzle，64B 128B以此类推
    > 3. wgmma operand 使用share memory时，share memory需要符合上述要求传入descriptor来描述该share memory，使wgmma立即core matrix在整个share memory的排布
3. 因为operand在share memory中因此选择wgmma时还需要选择core matrix在share memory中排布时K major还是M N major（core matrix连续的维度是哪一个）。对于flash attention2中mma1的v就可以反应出来，构造share memory时使用的是k major atom（这是为了使用tma传输时符合第0个维度连续的要求），但是wgmma atom的operand B选择的是MN major，这是因为wgmma使用的其实是Vt，此时以不同的layout看待这块share memory，因此变成了MN major
4. 因为是异步指令，因此需要提供相应的同步机制，这里的同步机制使用的是fence机制，具体可以参考https://research.colfax-intl.com/cutlass-tutorial-wgmma-hopper/ Synchronization for WGMMA小节，需要注意的是wgmma对于share memory是async proxy，因此与generic proxy的可见性一致需要使用cutlass::arch::fence_view_async_shared()（fence.proxy.async）来保证
### tma
tma 是 hopper架构新添加的一个用于global memory与share memory数据传输的硬件单元，他有这些优势：
1. 专有硬件单元用于传输，地址的计算完全由该硬件单元负责，因此寄存器使用减少
2. 发起指令时只需要一个thread
3. 支持一些新的特性，如reduce multicast

对于该硬件，编程模型相对于以前也有了变化：
1. 该硬件需要传入tensor map用于指定global tensor的layout以及share memory的layout，以及每次传输的box大小
2. kernel中指定传输哪一部分global tensor需要传入的是coordinate，因此引入了ArithTuples来解决，每个单元是一个坐标。使用local_tile切分后获得坐标的的逻辑是：local_tile切分ArithTuples后得到layout计算，指定某个tile获得global tensor起始的逻辑coordinate，再传入tma指令中，tma自己会通过tensor map信息来获得coordinate 到真正global memory address的转换
3. kernel中指定的share memory只需要传入起始的ptr，那他是怎么理解share memory的layout的呢？其实是在host构建tensor map的时候需要share memory，此时就获得了share memory的layout信息，调用tma指令是需要传入该tensor map的
4. 异步指令需要对应的同步机制。对于load来说，使用的是mbarrier，该barrier可以同时指定线程arrive数，以及传输数据大小，只有两个条件同时满足才行，同时对于发起tma指令的thread，在指令返回后自动arrive，不需要手动arrive，其他线程需要手动arrive；对于store来说，使用的是fence机制。具体可看https://research.colfax-intl.com/tutorial-hopper-tma/ 
* H100 255.154 -> 332.196

### tma store and stmatrix
1. 使用stmatrix将register O传输到share memory中，stmatrix的用法和ldmatrix一致，需要每个thread 拥有的fragment符合要求，其实是针对tensor core设计的，因为这个要求本来就是为tensor core thread fragment设定的。因为传输的的类型要求是fp16，因此做了一次类型转换。
2. 使用tma store将share memory O传输到global memory中，在构造tma store tensor
 map sO_tma的时候可以看到swizzle<3, 4, 3>，因为这里以每字节为单个单元，而在kernel中使用sO的swizzle<3, 3, 3>，这里以每个half为单个单元，两者是等价的
 * H100 332.196 -> 334.887
----
从现在开始测量标准改成warmup100，迭代1000（之前为warmup10，迭代100）
Approx. TFLOPS:332.151
### intra warp software pipeline
intra-warp software pipeline 在warp内进行了2 stage pipeline
1. 将$gemm 1_{i + 1} 和 softmax_{i}、gemm 2_{i}$进行了overlap，利用了wgmma异步完成的特点
2. 在一次迭代内完成这些操作$copyK_{i+2}、copyV_{i+1}、gemm 1_{i + 1} 和 softmax_{i}、gemm 2_{i}$

对于同步操作使用了cutlass中提供的PipelineTmaAsync配合PipelineState使用：
1. PipelineTmaAsync包含了k个stage的2个barrier，这两个barrier分为full barrier 和 empty barrier
2. 在初始化时需要指定很多param，包括该thread是producer还是consumer、producer的transaction bytes、consumer的arrive count
3. 编程为producer、consumer模型：对于producer来说，producer_acquire等待empty barrier到达指定的phase，表示producer可用；producer_commit 将full barrier的arrive count++，full barrier arrive count和transaction bytes到达指定值，phase翻转；consumer wait等待full barrier达到指定phase，表示consumer 可用；consumer release将empty barrier 的arrive count++，empty barrier arrive count到达指定值，phase翻转
4. 对于stage的记录（到达了哪个stage），即使用PipelineState，其是一个循环的计数值，==stage值时，index置为0，等待的phase翻转

刚开始的实现性能急剧下降Approx. TFLOPS:44.710，这是因为将mma1rP构造成了2 stage register fragment，后面指定使用哪一个stage register fragment的时候使用了动态寻址。但是寄存器不能被动态寻址，不能生成类似 “register_array[index]” 的机器指令。所以 ptxas 只能给它放一个 per-thread local backing storage，也就是说因为动态寻址的问题，这里的寄存器spill到了local memory中保存，因此插入了st ld local memory的指令，使wgmma的流水线中断，强制串行化。

sass指令WARPGROUP.DEPBAR的插入会中断wgmma的pipeline，使其串行化，这里插入的原因是每次hgmma的accumulator都是mma2rO，有依赖。

将mma1rP的两个stage分为两个独立的fragment后，寄存器不会spill到local memory中，但是$gemm 1_{i + 1} 和 softmax_{i}、gemm 2_{i}$似乎没有进行overlap
* H100 252.173

没有进行overlap的原因如下：
1. 设计中write_state.index() 和 read_state.index() 永远相反，但是这是我自己的逻辑，编译器是看不到的，ptxas 能看到的是两个独立 runtime 分支
2. 将softmax和gemm1的分支分开写，且分支判断条件分别是read_state.index()和write_state.index()，编译器是无法证明没有读同一个accumulator register，因此gemm1和softmax运行时逻辑上虽然不会使用同一份fragment但是编译器是不知道的，无法生成更好的overlap模式
3. 因此将两者放在同一个分支逻辑判断中，都使用write_state.index()进行判断
* H100 262.595
1. 这里性能依旧较低是因为用了较多的register导致了SM occupancy下降，从4 block/SM下降到3 block/SM
2. 现在的overlap模式是HGMMA、softmax 一段、DEPBAR、HGMMA、softmax 一段、DEPBAR、HGMMA ...相当于没有一次性issue很多HGMMA，而是在中间穿插了softmax

将2-stage overlap改成了$softmax_{i+1} 和 gemm2_{i}$进行overlap
1. 从ptx可以分析出进行了overlap，sass分析出overlap模式是HGMMA、softmax 一段、DEPBAR、HGMMA、softmax 一段、DEPBAR、HGMMA ...，这是由编译器决定的
2. 这个overlap模式降低了register pressure，达到了更高的occupancy 4 block/SM
* H100 308.460

调换了rescale rO的位置，使gemm1 和gemm 2可以连续issue，ptx层面两者是连续issue的，但是sass可以看到gemm2依旧穿插softmax，依旧是编译器调度的结果。编译器在隐藏 HGMMA latency
> 从调度器视角看，连续发射 gemm2 的 HGMMA 如果每一步都被 accumulator dependency 卡住，空周期会很多。把 softmax 的 MUFU.EX2 / FADD / FMUL / SHFL 塞进去，可以提高指令级并行度。但这不一定提高整体性能，因为它可能破坏你想要的宏观 pipeline
* H100 312.453

### warp specialization

### persistent kernel

In [ ]:
!make clean
!make -j4

In [ ]:
!make clean
!make -j$(nproc)
%set_env PYTHONPATH ./python
%set_env NEEDLE_BACKEND nd
!python3 -m pytest tests/project/test_flashattention.py -l -v -k "test_attention_activation_vs_torch and 64 and False and 0.0 and cuda and auto"

!python3 tests/project/benchmark.py \
  --batch_size 8 \
  --num_heads 12 \
  --seq_len 1024 \
  --head_dim 64 \
  --dropout 0.0 \
  --warmup 100 \
  --repeats 1000 \
  --kernel auto

In [ ]:
# ncu profile
!ncu --launch-skip 10 --launch-count 1 --set full \
  --kernel-name regex:flash_attention_kernel --kernel-name-base demangled \
  --import-source yes --source-folders /home/xyx/needle/src,/home/xyx/needle/python \
  -o /home/xyx/needle/flash_attention_profile_sOswizzle -f \
  python3 /home/xyx/needle/tests/project/benchmark.py \
  --batch_size 8 --num_heads 12 --seq_len 1024 --head_dim 64 --dropout 0.0